In [1]:
# ===============================
# Importação de bibliotecas
# ===============================

import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

In [2]:
# ===============================
# Carregamento dos dados - Olist (E-commerce)
# ===============================

orders = pd.read_csv("olist_orders_dataset.csv")          # Informações dos pedidos
items = pd.read_csv("olist_order_items_dataset.csv")      # Itens comprados em cada pedido
customers = pd.read_csv("olist_customers_dataset.csv")    # Dados dos clientes
payments = pd.read_csv("olist_order_payments_dataset.csv")# Informações de pagamento
products = pd.read_csv("olist_products_dataset.csv")      # Dados dos produtos
translation = pd.read_csv("product_category_name_translation.csv")  # Tradução das categorias

In [3]:
# ===============================
# Verificação dos dados - Orders
# ===============================

print("Amostra:")
print(orders.head().to_string(index=False))

print("\nValores nulos:")
print(orders.isnull().sum().to_string())

Amostra:
                        order_id                      customer_id order_status order_purchase_timestamp   order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7 9ef432eb6251297304e76186b10a928d    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15          2017-10-04 19:55:00           2017-10-10 21:25:13           2017-10-18 00:00:00
53cdb2fc8bc7dce0b6741e2150273451 b0830fb4747a6c6d20dea0b8c802d7ef    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27          2018-07-26 14:31:00           2018-08-07 15:27:45           2018-08-13 00:00:00
47770eb9100c2d0c44946d9cf07ec65d 41ce2a54c0b03bf3443c3d931a367089    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23          2018-08-08 13:50:00           2018-08-17 18:06:29           2018-09-04 00:00:00
949d5b44dbf5de918fe9c16f97b45f8a f88197465ea7920adcdbec7375364d82    delivered      2017-11-18 19:28:06 2017-11-18 19:45:59          2017-1

In [4]:
# ===============================
# Verificação dos dados - Customers
# ===============================

print("Amostra:")
print(customers.head().to_string(index=False))

print("\nValores nulos:")
print(customers.isnull().sum().to_string())

Amostra:
                     customer_id               customer_unique_id  customer_zip_code_prefix         customer_city customer_state
06b8999e2fba1a1fbc88172c00ba8bc7 861eff4711a542e4b93843c6dd7febb0                     14409                franca             SP
18955e83d337fd6b2def6b18a428ac77 290c77bc529b7ac935b93aa66c333dc3                      9790 sao bernardo do campo             SP
4e7b3e00288586ebd08712fdd0374a03 060e732b5b29e8181a18229c7b0b2b5e                      1151             sao paulo             SP
b2b6027bc5c5109e529d4dc6358b12c3 259dac757896d24d7702b9acbbff3f3c                      8775       mogi das cruzes             SP
4f2d8ab171c80ec8364f7c12e35b23ad 345ecd01c38d18a9036ed96c73b8d066                     13056              campinas             SP

Valores nulos:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0


In [5]:
# ===============================
# Verificação dos dados - Items
# ===============================

print("Amostra:")
print(items.head().to_string(index=False))

print("\nValores nulos:")
print(items.isnull().sum().to_string())

Amostra:
                        order_id  order_item_id                       product_id                        seller_id shipping_limit_date  price  freight_value
00010242fe8c5a6d1ba2dd792cb16214              1 4244733e06e7ecb4970a6e2683c13e61 48436dade18ac8b2bce089ec2a041202 2017-09-19 09:45:35  58.90          13.29
00018f77f2f0320c557190d7a144bdd3              1 e5f2d52b802189ee658865ca93d83a8f dd7ddc04e1b6c2c614352b383efe2d36 2017-05-03 11:05:13 239.90          19.93
000229ec398224ef6ca0657da4fc703e              1 c777355d18b72b67abbeef9df44fd0fd 5b51032eddd242adc84c38acab88f23d 2018-01-18 14:48:30 199.00          17.87
00024acbcdf0a6daa1e931b038114c75              1 7634da152a4610f1595efa32f14722fc 9d7a1d34a5052409006425275ba1c2b4 2018-08-15 10:10:18  12.99          12.79
00042b26cf59d7ce69dfabb4e55b4fd9              1 ac6c3623068f30de03045865e4e10089 df560393f3a51e74553ab94004ba5c87 2017-02-13 13:57:51 199.90          18.14

Valores nulos:
order_id               0
order_item_id 

In [6]:
# ===============================
# Verificação dos dados - Payments
# ===============================

print("Amostra:")
print(payments.head().to_string(index=False))

print("\nValores nulos:")
print(payments.isnull().sum().to_string())

Amostra:
                        order_id  payment_sequential payment_type  payment_installments  payment_value
b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card                     8          99.33
a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card                     1          24.39
25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card                     1          65.71
ba78997921bbcdc1373bb41e913ab953                   1  credit_card                     8         107.78
42fdf880ba16b47b59251dd489d4441a                   1  credit_card                     2         128.45

Valores nulos:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0


In [7]:
# ===============================
# Verificação dos dados - Products
# ===============================

print("Amostra:")
print(products.head().to_string(index=False))

print("\nValores nulos:")
print(products.isnull().sum().to_string())

Amostra:
                      product_id product_category_name  product_name_lenght  product_description_lenght  product_photos_qty  product_weight_g  product_length_cm  product_height_cm  product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5            perfumaria                 40.0                       287.0                 1.0             225.0               16.0               10.0              14.0
3aa071139cb16b67ca9e5dea641aaa2f                 artes                 44.0                       276.0                 1.0            1000.0               30.0               18.0              20.0
96bd76ec8810374ed1b65e291975717f         esporte_lazer                 46.0                       250.0                 1.0             154.0               18.0                9.0              15.0
cef67bcfe19066a932b7673e239eb23d                 bebes                 27.0                       261.0                 1.0             371.0               26.0                4.0              26.0
9

In [8]:
# ===============================
# Verificação dos dados - Translation
# ===============================

print("Amostra:")
print(translation.head().to_string(index=False))

print("\nValores nulos:")
print(translation.isnull().sum().to_string())

Amostra:
 product_category_name product_category_name_english
          beleza_saude                 health_beauty
informatica_acessorios         computers_accessories
            automotivo                          auto
       cama_mesa_banho                bed_bath_table
      moveis_decoracao               furniture_decor

Valores nulos:
product_category_name            0
product_category_name_english    0


In [9]:
# ===============================
# Carregamento dos dados - Telco (Churn)
# ===============================

telco = pd.read_csv("Telco-Customer-Churn.csv")  # Dataset de clientes com churn

In [10]:
# ===============================
# Limpeza e tratamento dos dados - Telco
# ===============================

# Converter a coluna TotalCharges para numérico
# (valores inválidos viram NaN)

telco["TotalCharges"] = pd.to_numeric(
    telco["TotalCharges"], errors="coerce"
)

# Remover registros com valores nulos nessa coluna

telco = telco.dropna(subset=["TotalCharges"])

In [11]:
# ===============================
# Verificação de valores nulos após tratamento
# ===============================

print("Valores nulos após tratamento:")
print(telco.isnull().sum().to_frame(name="Valores Nulos"))

Valores nulos após tratamento:
                  Valores Nulos
customerID                    0
gender                        0
SeniorCitizen                 0
Partner                       0
Dependents                    0
tenure                        0
PhoneService                  0
MultipleLines                 0
InternetService               0
OnlineSecurity                0
OnlineBackup                  0
DeviceProtection              0
TechSupport                   0
StreamingTV                   0
StreamingMovies               0
Contract                      0
PaperlessBilling              0
PaymentMethod                 0
MonthlyCharges                0
TotalCharges                  0
Churn                         0


In [12]:
# ===============================
# Integração dos dados - Olist
# ===============================

# Merge orders + customers
df_olist = orders.merge(customers, on="customer_id", how="left")

# Merge com items
df_olist = df_olist.merge(items, on="order_id", how="left")

# Merge com payments
df_olist = df_olist.merge(payments, on="order_id", how="left")

# Merge com products
df_olist = df_olist.merge(products, on="product_id", how="left")

# Merge com tradução de categorias
df_olist = df_olist.merge(
    translation,
    on="product_category_name",
    how="left"
)

In [13]:
# ===============================
# Verificação após integração
# ===============================

print("Amostra:")
print(df_olist.head().to_string(index=False))

print("\nDimensão do dataset:")
print(df_olist.shape)

Amostra:
                        order_id                      customer_id order_status order_purchase_timestamp   order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date               customer_unique_id  customer_zip_code_prefix customer_city customer_state  order_item_id                       product_id                        seller_id shipping_limit_date  price  freight_value  payment_sequential payment_type  payment_installments  payment_value product_category_name  product_name_lenght  product_description_lenght  product_photos_qty  product_weight_g  product_length_cm  product_height_cm  product_width_cm product_category_name_english
e481f51cbdc54678b7cc49136f2d6af7 9ef432eb6251297304e76186b10a928d    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15          2017-10-04 19:55:00           2017-10-10 21:25:13           2017-10-18 00:00:00 7c396fd4830fd04220f754e42b4e5bff                      3149     sao paulo             SP  

In [14]:
# ===============================
# Criação de base por cliente
# ===============================

df_cliente = df_olist.groupby("customer_unique_id").agg({
    "order_id": "nunique",
    "price": "sum",
    "payment_value": "sum",
    "order_purchase_timestamp": ["min", "max"]
}).reset_index()

df_cliente.columns = [
    "customer_unique_id",
    "total_pedidos",
    "total_gasto",
    "total_pago",
    "primeira_compra",
    "ultima_compra"
]

In [15]:
df_cliente["primeira_compra"] = pd.to_datetime(df_cliente["primeira_compra"])
df_cliente["ultima_compra"] = pd.to_datetime(df_cliente["ultima_compra"])

In [16]:
# ===============================
# Criação da variável churn
# ===============================

data_ref = df_cliente["ultima_compra"].max()

df_cliente["dias_sem_compra"] = (
    data_ref - df_cliente["ultima_compra"]
).dt.days

df_cliente["churn"] = df_cliente["dias_sem_compra"].apply(
    lambda x: 1 if x > 90 else 0
)

In [17]:
# ===============================
# Distribuição de churn
# ===============================

churn_counts = df_cliente["churn"].value_counts()
total = churn_counts.sum()

print("Distribuição de churn:")
print(f"Churn: {churn_counts.get(1,0)} ({churn_counts.get(1,0)/total:.2%})")
print(f"Ativos: {churn_counts.get(0,0)} ({churn_counts.get(0,0)/total:.2%})")

Distribuição de churn:
Churn: 86432 (89.94%)
Ativos: 9664 (10.06%)


In [18]:
# ===============================
# Análise - Gasto médio por churn
# ===============================

media_gasto = df_cliente.groupby("churn")["total_gasto"].mean()

print("Gasto médio:")
print(f"Ativos: {media_gasto.get(0):.2f}")
print(f"Churn: {media_gasto.get(1):.2f}")

Gasto médio:
Ativos: 144.86
Churn: 148.20


In [19]:
# ===============================
# Análise - Frequência de compra
# ===============================

media_pedidos = df_cliente.groupby("churn")["total_pedidos"].mean()

print("Média de pedidos:")
print(f"Ativos: {media_pedidos.get(0):.2f}")
print(f"Churn: {media_pedidos.get(1):.2f}")

Média de pedidos:
Ativos: 1.04
Churn: 1.03


In [20]:
# ===============================
# Criação da tabela de transações
# ===============================

df_transacoes = df_olist[[
    "customer_unique_id",
    "order_id",
    "order_purchase_timestamp",
    "price",
    "payment_value",
    "product_category_name_english"
]].copy()

In [21]:
# ===============================
# Registro das tabelas no DuckDB
# ===============================

con = duckdb.connect()

con.register("df_cliente", df_cliente)
con.register("df_transacoes", df_transacoes)
con.register("df_olist", df_olist)
con.register("telco", telco)

print("Tabelas criadas no DuckDB com sucesso")

Tabelas criadas no DuckDB com sucesso


In [22]:
# ===============================
# QUERY 1 — Churn por perfil de gasto
# ===============================

q1 = con.execute("""
    WITH perfil AS (
        SELECT
            customer_unique_id,
            total_pedidos,
            total_gasto,
            dias_sem_compra,
            Churn,
            CASE
                WHEN total_gasto < 100 THEN 'Baixo Valor'
                WHEN total_gasto BETWEEN 100 AND 500 THEN 'Médio Valor'
                ELSE 'Alto Valor'
            END AS perfil_gasto
        FROM df_cliente
    )
    SELECT
        perfil_gasto,
        Churn,
        COUNT(*) AS total_clientes,
        ROUND(AVG(total_gasto), 2) AS gasto_medio,
        ROUND(AVG(total_pedidos), 2) AS pedidos_medio,
        ROUND(100.0 * SUM(CASE WHEN Churn = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS taxa_churn
    FROM perfil
    GROUP BY perfil_gasto, Churn
    ORDER BY perfil_gasto, Churn
""").df()

print(q1.to_string(index=False))

perfil_gasto  churn  total_clientes  gasto_medio  pedidos_medio  taxa_churn
  Alto Valor      0             390       999.36           1.18         0.0
  Alto Valor      1            3690       971.51           1.11       100.0
 Baixo Valor      0            5501        52.27           1.02         0.0
 Baixo Valor      1           48778        52.97           1.01       100.0
 Médio Valor      0            3773       191.53           1.07         0.0
 Médio Valor      1           33964       195.52           1.06       100.0


In [23]:
# ===============================
# QUERY 2 — Ticket Médio por Churn
# ===============================

q2 = con.execute("""
    SELECT
        Churn,
        COUNT(*) AS total_clientes,
        ROUND(SUM(total_gasto)) AS receita_total,
        ROUND(SUM(total_gasto) / NULLIF(SUM(total_pedidos), 0), 2) AS ticket_medio
    FROM df_cliente
    GROUP BY Churn
    ORDER BY Churn
""").df()

print(q2.to_string(index=False))

 churn  total_clientes  receita_total  ticket_medio
     0            9664      1399923.0        138.81
     1           86432     12809327.0        143.35


In [24]:
# ===============================
# QUERY 3 — Consumo por categoria
# ===============================

q3 = con.execute("""
    SELECT
        product_category_name_english AS categoria,
        COUNT(DISTINCT order_id) AS total_pedidos,
        ROUND(SUM(price), 2) AS receita_total,
        ROUND(AVG(price), 2) AS ticket_medio_categoria,
        ROUND(
            100.0 * SUM(price) / SUM(SUM(price)) OVER(),
            2
        ) AS participacao_receita
    FROM df_transacoes
    WHERE product_category_name_english IS NOT NULL
    GROUP BY categoria
    ORDER BY receita_total DESC
""").df()

print(q3.to_string(index=False))

                              categoria  total_pedidos  receita_total  ticket_medio_categoria  participacao_receita
                          health_beauty           8836     1297490.77                  130.07                  9.26
                          watches_gifts           5624     1253143.30                  202.09                  8.94
                         bed_bath_table           9417     1092551.02                   92.41                  7.80
                         sports_leisure           7720     1023996.34                  114.48                  7.31
                  computers_accessories           6689      942277.57                  116.59                  6.72
                        furniture_decor           6449      765093.89                   87.50                  5.46
                             housewares           5884      666587.00                   90.63                  4.76
                             cool_stuff           3632      662309.49   

In [25]:
# ===============================
# QUERY 4 — Frequência de compra e ranking (quartis)
# ===============================

q4 = con.execute("""
    SELECT
        quartil_frequencia,
        churn,
        COUNT(*) AS total_clientes,
        ROUND(AVG(total_pedidos), 2) AS pedidos_medio,
        ROUND(AVG(total_gasto), 2) AS gasto_medio
    FROM (
        SELECT
            customer_unique_id,
            total_pedidos,
            total_gasto,
            churn,
            NTILE(4) OVER(ORDER BY total_pedidos DESC) AS quartil_frequencia
        FROM df_cliente
    ) t
    GROUP BY quartil_frequencia, churn
    ORDER BY quartil_frequencia, churn
""").df()

print(q4.to_string(index=False))

 quartil_frequencia  churn  total_clientes  pedidos_medio  gasto_medio
                  1      0            2424           1.17       155.56
                  1      1           21600           1.14       161.32
                  2      0            2454           1.00       139.81
                  2      1           21570           1.00       144.29
                  3      0            2333           1.00       137.13
                  3      1           21691           1.00       143.75
                  4      0            2453           1.00       146.69
                  4      1           21571           1.00       143.46


In [27]:
# QUERY 5 - Churn por faixa de renda

q5 = con.execute("""
    WITH faixas AS (
        SELECT
            customerID,
            LOWER(Churn) AS churn,
            MonthlyCharges,

            TRY_CAST(TotalCharges AS DECIMAL) AS TotalCharges,
            tenure,

            CASE
                WHEN MonthlyCharges < 30 THEN 'Baixa Renda'
                WHEN MonthlyCharges < 70 THEN 'Média Renda'
                ELSE 'Alta Renda'
            END AS faixa_renda
        FROM telco
    )

    SELECT
        faixa_renda,
        churn,
        COUNT(*) AS total,
        ROUND(AVG(tenure), 1) AS tempo_medio_meses,
        ROUND(AVG(TotalCharges), 2) AS gasto_medio
    FROM faixas
    GROUP BY faixa_renda, churn
    ORDER BY faixa_renda, churn
""").df()

print(q5.to_string(index=False))

faixa_renda churn  total  tempo_medio_meses  gasto_medio
 Alta Renda    no   2315               45.2      4228.55
 Alta Renda   yes   1274               21.5      2019.98
Baixa Renda    no   1485               31.5       688.60
Baixa Renda   yes    162                7.3       161.51
Média Renda    no   1363               31.5      1747.32
Média Renda   yes    433               11.7       608.10


In [28]:
# ===============================
# Estatística descritiva por churn
# ===============================

estatisticas = df_cliente.groupby("churn").agg({
    "total_gasto": ["mean", "median", "std"],
    "total_pedidos": ["mean", "median", "std"],
    "dias_sem_compra": ["mean", "median", "std"]
})

print(estatisticas)


# ===============================
# Insight inicial
# ===============================

# Clientes em churn apresentam maior tempo sem compra
# e menor frequência de pedidos, indicando perda de engajamento.

      total_gasto                    total_pedidos                   \
             mean median         std          mean median       std   
churn                                                                 
0      144.859604   86.0  241.329981      1.043564    1.0  0.283869   
1      148.201211   89.9  248.756322      1.033830    1.0  0.205136   

      dias_sem_compra                     
                 mean median         std  
churn                                     
0           71.584541   72.0   10.737741  
1          311.903647  291.0  142.641831  


In [29]:
# ===============================
# Visualizando os dados - Proporção de Churn
# ===============================

import plotly.express as px

df_churn_plot = pd.DataFrame({
    "Status": ["Ativo", "Churn"],
    "Quantidade": [churn_counts.get(0, 0), churn_counts.get(1, 0)]
})

fig = px.bar(df_churn_plot,
             x="Status",
             y="Quantidade",
             color="Status",
             color_discrete_map={"Ativo": "#2ecc71", "Churn": "#e74c3c"},
             title="Proporção de Churn — df_cliente",
             text="Quantidade")

fig.update_traces(width=0.4, textfont_color="white", textposition="inside")
fig.update_layout(
     showlegend=False,
      xaxis=dict(range=[-0.5, 1.5])
 )

fig.show()

In [30]:
# ===============================
# Distribuição de Gasto por Churn (Clientes que cancelam gastam menos?)
# ===============================

fig = px.box(df_cliente,
             x="churn",
             y="total_gasto",
             color="churn",
             color_discrete_map={0: "#2ecc71", 1: "#e74c3c"},
             title="Distribuição de Gasto por Grupo de Churn",
             labels={"churn": "Status", "total_gasto": "Total Gasto (R$)"},
             category_orders={"churn": [0, 1]})

fig.update_layout(
    showlegend=False,
    yaxis=dict(range=[0, 1000])
)
fig.show()

**Distribuição de Gasto por Grupo de Churn**


O boxplot revela que a distribuição de gasto entre clientes ativos e em churn é muito semelhante, com mediana em torno de R\$ 80-100 para ambos os grupos. Clientes ativos apresentam uma leve tendência a gastar mais, porém a diferença não é expressiva. Ambos os grupos possuem outliers com gastos muito acima da média, chegando a R$ 13.440,00.



In [31]:
# ===============================
# Tempo sem compra — Principal indicador de churn
# ===============================

import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Clientes Ativos", "Clientes em Churn"))

fig.add_trace(
    go.Histogram(x=df_cliente[df_cliente["churn"] == 0]["dias_sem_compra"],
                 nbinsx=30,
                 marker_color="#2ecc71",
                 marker_line=dict(color="white", width=1),
                 name="Ativo"),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=df_cliente[df_cliente["churn"] == 1]["dias_sem_compra"],
                 nbinsx=30,
                 marker_color="#e74c3c",
                 marker_line=dict(color="white", width=1),
                 name="Churn"),
    row=1, col=2
)

fig.update_layout(title="Distribuição de Dias sem Compra por Grupo",
                  showlegend=False)
fig.update_xaxes(title_text="Dias sem Compra")
fig.update_yaxes(title_text="Quantidade de Clientes")

fig.show()

**Distribuição de Dias sem Compra por Grupo**

Clientes ativos ficaram entre 45 e 90 dias sem comprar, com concentração entre 60 e 80 dias. Já os clientes em churn apresentam uma distribuição mais ampla, com maioria entre 100 e 300 dias sem comprar, indicando que o abandono acontece gradualmente ao longo do tempo.


In [32]:
# ===============================
# Gasto médio por grupo de churn
# ===============================

df_gasto_medio = df_cliente.groupby("churn")["total_gasto"].mean().reset_index()
df_gasto_medio["churn"] = df_gasto_medio["churn"].map({0: "Ativo", 1: "Churn"})
df_gasto_medio["total_gasto"] = df_gasto_medio["total_gasto"].round(2)

fig = px.bar(df_gasto_medio,
             x="churn",
             y="total_gasto",
             color="churn",
             color_discrete_map={"Ativo": "#2ecc71", "Churn": "#e74c3c"},
              title="Clientes que cancelam gastam menos? Comparação de gasto médio",
             labels={"churn": "Status do Cliente", "total_gasto": "Gasto Médio (R$)"},
             text="total_gasto")

fig.update_traces(width=0.4, texttemplate="R$ %{text}", textposition="inside")
fig.update_layout(showlegend=False, yaxis_title="Gasto Médio (R$)",
    xaxis_title="")
fig.show()



 Clientes em churn apresentam padrão de gasto diferente dos ativos,
 indicando que o nível de consumo está relacionado à retenção.

In [33]:

# ===============================
# Estatítica Descritiva Dataset Telco - Contratos
# ===============================

# Tratamento da coluna TotalCharges

telco["TotalCharges"] = pd.to_numeric(
    telco["TotalCharges"],
    errors="coerce"
)

telco = telco.dropna(subset=["TotalCharges"])

# Separando clientes ativos e em churn
ativos = telco[telco["Churn"] == "No"]
churn  = telco[telco["Churn"] == "Yes"]

# Colunas numéricas que vamos analisar
colunas = ["tenure", "MonthlyCharges", "TotalCharges"]

# Calculando média, mediana e desvio padrão para cada grupo
for col in colunas:
    print(f"\n {col}")
    print(f"  Ativos   Média: {ativos[col].mean():.2f} | Mediana: {ativos[col].median():.2f} | Desvio: {ativos[col].std():.2f}")
    print(f"  Churn    Média: {churn[col].mean():.2f}  | Mediana: {churn[col].median():.2f}  | Desvio: {churn[col].std():.2f}")


 tenure
  Ativos   Média: 37.65 | Mediana: 38.00 | Desvio: 24.08
  Churn    Média: 17.98  | Mediana: 10.00  | Desvio: 19.53

 MonthlyCharges
  Ativos   Média: 61.31 | Mediana: 64.45 | Desvio: 31.09
  Churn    Média: 74.44  | Mediana: 79.65  | Desvio: 24.67

 TotalCharges
  Ativos   Média: 2555.34 | Mediana: 1683.60 | Desvio: 2329.46
  Churn    Média: 1531.80  | Mediana: 703.55  | Desvio: 1890.82


In [34]:
# ===============================
# Boxplot - Estatística Descritiva Telco
# ===============================
import plotly.express as px

# Lista de colunas para visualizar
colunas = ["tenure", "MonthlyCharges", "TotalCharges"]

# Gerando um boxplot para cada coluna
for col in colunas:
    fig = px.box(telco,
                 x="Churn",
                 y=col,
                 color="Churn",
                 color_discrete_map={"No": "#2ecc71", "Yes": "#e74c3c"},
                 title=f"Distribuição de {col} por Grupo de Churn",
                 labels={"Churn": "Status", col: col})

    fig.update_layout(showlegend=False)
    fig.show()



**Clientes que cancelam têm perfil diferente?**

Sim, os dados mostram:
- Clientes que ficam menos tempo: 18 meses em média (ativos ficam 37 meses)
- Clientes que pagam mais caro: R\$ 74,00/mês (ativos pagam R$ 61/mês)

 Clientes novos, com mensalidades altas e pouco tempo de casa são os que mais cancelam. Isso sugere que o preço elevado logo no início pode ser o principal gatilho do churn.

In [35]:
# ===============================
# Churn por Tipo de Contrato - Dataset Telco
# ===============================
import plotly.express as px

# Contando churn por tipo de contrato
contrato = telco.groupby(["Contract", "Churn"]).size().reset_index(name="quantidade")

# Gráfico de barras agrupadas
fig = px.bar(contrato,
             x="Contract",
             y="quantidade",
             color="Churn",
             color_discrete_map={"No": "#2ecc71", "Yes": "#e74c3c"},
             title="Churn por Tipo de Contrato",
             labels={"Contract": "Tipo de Contrato", "quantidade": "Nº de Clientes", "Churn": "Status"},
             barmode="group",
             text="quantidade")

fig.update_traces(textposition="outside")
fig.update_layout(xaxis_title="Tipo de Contrato", yaxis_title="Nº de Clientes")
fig.show()

**Quem cancela mais? O tipo de contrato influencia no churn?**

Sim, é o fator mais decisivo! Clientes com contrato mensal (Month-to-month) são os que mais cancelam: 1.655 churns, muito acima dos contratos anuais (166) e de 2 anos (48).

Quanto maior o compromisso com o contrato, menor o churn. Clientes mensais têm liberdade de cancelar a qualquer momento.

In [36]:
# ===============================
# Churn por Método de Pagamento - Dataset Telco
# ===============================
import plotly.express as px

# Contando churn por método de pagamento
pagamento = telco.groupby(["PaymentMethod", "Churn"]).size().reset_index(name="quantidade")

# Gráfico de barras agrupadas
fig = px.bar(pagamento,
             x="PaymentMethod",
             y="quantidade",
             color="Churn",
             color_discrete_map={"No": "#2ecc71", "Yes": "#e74c3c"},
             title="Churn por Método de Pagamento",
             labels={"PaymentMethod": "Método de Pagamento", "quantidade": "Nº de Clientes", "Churn": "Status"},
             barmode="group",
             text="quantidade")

fig.update_traces(textposition="outside")
fig.update_layout(xaxis_title="", yaxis_title="Nº de Clientes")
fig.show()

**O método de pagamento influencia no cancelamento?**

Sim, o gráfico acima revela que  electronic check (Boletos eletrônico) se destaca com 1.071 cancelamentos, muito acima dos outros métodos

Bank transfer (Débito automático): 258

Credit card (cartão de crédito): 232

Mailed check (Boleto físico): 308.

Clientes que pagam por boletos eletrônicos cancelam muito mais. Esse método é geralmente usado por quem ainda não tem vínculo automático com a operadora, o que facilita o cancelamento.

In [37]:
# ===============================
# Probabilidade Condicional - Telco
# Qual perfil possui maior risco de churn?
# ===============================

# Função para calcular P(churn | condição)

def prob_churn(coluna, valor):

    grupo = telco[telco[coluna] == valor]

    probabilidade = (
        grupo["Churn"]
        .value_counts(normalize=True)
        .get("Yes", 0)
    )

    total_clientes = len(grupo)

    print(f"{valor}")
    print(f"Clientes: {total_clientes}")
    print(f"P(churn | {valor}) = {probabilidade:.2%}")
    print("-" * 40)

    print("Probabilidade Condicional de Churn\n")

# ===============================
# Tipo de contrato
# ===============================

print(" Tipos de contrato:\n")

prob_churn("Contract", "Month-to-month")
prob_churn("Contract", "One year")
prob_churn("Contract", "Two year")

# ===============================
# Método de pagamento
# ===============================

print("\nMétodo de Pagamento:\n")

prob_churn("PaymentMethod", "Electronic check")
prob_churn("PaymentMethod", "Credit card (automatic)")

# ===============================
# Tipo de internet
# ===============================

print("\nTipo de Internet:\n")

prob_churn("InternetService", "Fiber optic")
prob_churn("InternetService", "DSL")

 Tipos de contrato:

Month-to-month
Clientes: 3875
P(churn | Month-to-month) = 42.71%
----------------------------------------
Probabilidade Condicional de Churn

One year
Clientes: 1472
P(churn | One year) = 11.28%
----------------------------------------
Probabilidade Condicional de Churn

Two year
Clientes: 1685
P(churn | Two year) = 2.85%
----------------------------------------
Probabilidade Condicional de Churn


Método de Pagamento:

Electronic check
Clientes: 2365
P(churn | Electronic check) = 45.29%
----------------------------------------
Probabilidade Condicional de Churn

Credit card (automatic)
Clientes: 1521
P(churn | Credit card (automatic)) = 15.25%
----------------------------------------
Probabilidade Condicional de Churn


Tipo de Internet:

Fiber optic
Clientes: 3096
P(churn | Fiber optic) = 41.89%
----------------------------------------
Probabilidade Condicional de Churn

DSL
Clientes: 2416
P(churn | DSL) = 19.00%
----------------------------------------
Probabili

Os dados mostram que clientes com contratos mensais tendem a cancelar mais rápido, principalmente aqueles que usam Electronic Check e internet Fiber Optic. Isso indica um perfil menos fidelizado e mais sensível à experiência e ao custo do serviço.

Já clientes com contratos de longo prazo apresentam baixo churn, mostrando que aumentar fidelização pode ser uma estratégia eficiente para retenção.

In [38]:
# ===============================
# Feature Engineering
# Criação de novas variáveis
# ===============================

# Ticket médio
df_cliente["ticket_medio"] = (
    df_cliente["total_gasto"] /
    df_cliente["total_pedidos"]
)

# Frequência de compra
df_cliente["frequencia_compra"] = (
    df_cliente["total_pedidos"] /
    (df_cliente["dias_sem_compra"] + 1)
)

In [39]:
# ===============================
# Variáveis preditoras e alvo
# ===============================

X = df_cliente[
    [
        "total_gasto",
        "total_pedidos",
        "dias_sem_compra",
        "ticket_medio",
        "frequencia_compra"
    ]
]

y = df_cliente["churn"]

In [40]:
# ===============================
# Separação treino e teste
# ===============================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [41]:
# ===============================
# Treinamento do modelo
# Logistic Regression
# ===============================

modelo = LogisticRegression(max_iter=1000)

modelo.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [42]:
# ===============================
# Realizando previsões
# ===============================

y_pred = modelo.predict(X_test)

y_prob = modelo.predict_proba(X_test)[:,1]

In [43]:
# ===============================
# Avaliação do modelo
# ===============================

print("Accuracy:")
print(round(accuracy_score(y_test, y_pred), 4))

print("\nPrecision:")
print(round(precision_score(y_test, y_pred), 4))

print("\nRecall:")
print(round(recall_score(y_test, y_pred), 4))

print("\nMatriz de Confusão:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
1.0

Precision:
1.0

Recall:
1.0

Matriz de Confusão:
[[ 1917     0]
 [    0 17303]]


**Accuracy**

Percentual total de acertos.

**Precision**

Dos clientes previstos como churn:
quantos realmente eram churn.

**Recall**

Dos clientes que realmente cancelaram:
quantos o modelo conseguiu identificar.


In [44]:
# ===============================
# Score de risco de churn
# ===============================

df_cliente["score_risco"] = (
    modelo.predict_proba(X)[:,1] * 100
).round(0)

In [45]:
# ===============================
# Classificação de risco
# ===============================

def faixa_risco(score):

    if score >= 80:
        return "Crítico"

    elif score >= 60:
        return "Alto"

    elif score >= 40:
        return "Médio"

    else:
        return "Baixo"

df_cliente["faixa_risco"] = (
    df_cliente["score_risco"]
    .apply(faixa_risco)
)

In [46]:
# ===============================
# Recomendações estratégicas
# ===============================

def recomendacao(score):

    if score >= 80:
        return "Oferta agressiva de retenção"

    elif score >= 60:
        return "Contato preventivo"

    elif score >= 40:
        return "Campanha de engajamento"

    else:
        return "Cliente estável"

df_cliente["recomendacao"] = (
    df_cliente["score_risco"]
    .apply(recomendacao)
)

In [47]:
# ===============================
# Resultado final
# ===============================

df_cliente[
    [
        "customer_unique_id",
        "score_risco",
        "faixa_risco",
        "recomendacao"
    ]
].head()

,customer_unique_id,score_risco,faixa_risco,recomendacao
0,0000366f3b9a7992bf8c76cfdf3221e2,100.0,Crítico,Oferta agressiva de retenção
1,0000b849f77a49e4a4ce2b2a4ca5be3f,100.0,Crítico,Oferta agressiva de retenção
2,0000f46a3911fa3c0805444483337064,100.0,Crítico,Oferta agressiva de retenção
3,0000f6ccb0745a6a4b88665a16c9f078,100.0,Crítico,Oferta agressiva de retenção
4,0004aac84e0df4da2b147fca70cf8255,100.0,Crítico,Oferta agressiva de retenção


Foi desenvolvido um modelo de Machine Learning com o objetivo de prever a probabilidade de churn dos clientes, permitindo identificar perfis com maior risco de evasão.

Foram gerados:

- Score de risco (0–100)

- Classificação de risco

- Recomendações estratégicas de retenção

Isso permite apoio à tomada de decisão de forma preditiva e orientada por dados.